CSV -> Read Data -> Transform Data -> Write Data -> Parquet

## Reading the CSV files from Data Lake using Spark Dataframe Reader

References
Spark methods:
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.csv.html#pyspark.sql.DataFrameReader.csv
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.types.StructType.html#pyspark.sql.types.StructType
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.types.StructField.html#pyspark.sql.types.StructField

Spark data types:
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html?highlight=data%20types

Spark functions:
- https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.current_timestamp.html?highlight=current_timestamp

In [0]:
# Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import current_timestamp

In [0]:
databricks_secret_scope_name = "de-databricks-scope"
client_id_name = "de-databricks-entra-id-client-id"
tenant_id_name = "de-databricks-entra-id-tenant-id"
client_secret_name = "de-databricks-entra-id-client-secret"

client_id = dbutils.secrets.get(scope = databricks_secret_scope_name, key = client_id_name)
tenant_id = dbutils.secrets.get(scope = databricks_secret_scope_name, key = tenant_id_name)
client_secret = dbutils.secrets.get(scope = databricks_secret_scope_name, key = client_secret_name)

storage_account_name = "dedatabricksdl2"
source_container_name = "bronze"
target_container_name = "silver"

data_name = "circuits"
source_data_format = "csv"
target_data_format = "parquet"

In [0]:
# Bronze files path
source_path = f"abfss://{source_container_name}@{storage_account_name}.dfs.core.windows.net/{data_name}.{source_data_format}"
target_path = f"abfss://{target_container_name}@{storage_account_name}.dfs.core.windows.net/{data_name}"

### Transforming Circuits Data

In [0]:
circuits_schema = StructType(fields=[StructField("circuitId", IntegerType(), False),
                                     StructField("circuitRef", StringType(), True),
                                     StructField("name", StringType(), True),
                                     StructField("location", StringType(), True),
                                     StructField("country", StringType(), True),
                                     StructField("lat", DoubleType(), True),
                                     StructField("lng", DoubleType(), True),
                                     StructField("alt", IntegerType(), True),
                                     StructField("url", StringType(), True)
])

In [0]:
# Create a SparkSession
spark = SparkSession.builder\
    .appName("bronze_to_silver")\
    .getOrCreate()

In [0]:
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
# We won't use inferSchema because it will slow the process if our data is big
circuits = spark.read\
    .schema(circuits_schema)\
    .format("csv")\
    .option("header", "true")\
    .load(source_path)

#### Select the only columns we need

In [0]:
circuits = circuits.select("circuitId", "circuitRef", "name", "location", "country", "lat", "lng", "alt")

#### Rename some columns

In [0]:
circuits = circuits.withColumnRenamed("circuitId", "circuit_id")\
  .withColumnRenamed("circuitRef", "circuit_ref")\
  .withColumnRenamed("lat", "latitude")\
  .withColumnRenamed("lng", "longitude")\
  .withColumnRenamed("alt", "altitude") 

#### Get ingestion date by using current time

In [0]:
circuits = circuits.withColumn("ingestion_date", current_timestamp())

#### Write our data to target layer

In [0]:
circuits.write.mode("overwrite").format(source_data_format).save(target_path)